In [9]:
import pandas as pd

In [10]:
from google.cloud import bigquery
client = bigquery.Client()

query = '''
WITH FilteredData AS (
  SELECT
    *,
    SAFE_CAST(Valuation_Value AS FLOAT64) AS Numeric_Valuation
  FROM
    `proj-docai-dev.sales_rec_demo.cc_data_total`
  WHERE
    SAFE_CAST(Valuation_Value AS FLOAT64) > 1000000
    AND Stage IN ("Biddate Set", "Construction Documents", "General Contractor Award", "Low Bids Announced", "SUBBIDS: ASAP", "Construction Underway", "Post Bid")
    AND ParentCategories_PrimaryCategoryName IN (
      "Airport", "Apartments", "Auditoriums", "Bank", "College, University", "Condominiums", "Courthouses", "Dormitories", "Elementary, Pre Schools", "Fire and Police Stations", "Food Stores", "Government - Misc. Bldgs.", "Government Offices", "High Schools", "Hospitals, Clinics", "Hotels", "Junior High Schools", "Libraries", "Medical Offices", "Military - Misc.", "Military Housing", "Military Offices", "Museums", "Nursing Homes", "Offices", "Post Offices", "Prisons", "Religious Auditoriums", "Rental Warehouses", "Restaurants", "Retail Stores", "Shopping Centers", "Special, Vocational Schools", "Sports Arenas/Convention Centers", "Warehouses", "Athletic Bldgs", "Automotive", "Cafeterias", "Clubs, Community Centers", "Entertainment", "Golf Course / Country Club", "Laboratories", "Transportation Terminals", "Water and Sewage Treatment Plants"
    )
    AND EXISTS (
      SELECT 1
      FROM UNNEST(ParentCategories_ParentCategory) AS pc
      WHERE
        (pc._Name = "CIVIL" AND EXISTS (SELECT 1 FROM UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) AS sub WHERE sub IN ("Airport", "Offices", "Transportation Terminals", "Rental Warehouses", "Laboratories", "Clubs, Community Centers", "Golf Course / Country Club", "Broadcast Studios", "Museums", "Sports Arenas/Convention Centers", "Libraries", "Auditoriums", "Religious Auditoriums")))
        OR (pc._Name = "COMMERCIAL" AND EXISTS (SELECT 1 FROM UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) AS sub WHERE sub IN ("Airport", "Offices", "Transportation Terminals", "Rental Warehouses", "Laboratories")))
        OR (pc._Name = "COMMUNITY" AND EXISTS (SELECT 1 FROM UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) AS sub WHERE sub IN ("Clubs, Community Centers", "Transportation Terminals", "Golf Course / Country Club", "Broadcast Studios", "Laboratories", "Museums", "Offices", "Sports Arenas/Convention Centers", "Rental Warehouses", "Libraries", "Auditoriums", "Religious Auditoriums", "Airport")))
        OR (pc._Name = "EDUCATIONAL" AND EXISTS (SELECT 1 FROM UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) AS sub WHERE sub IN ("Elementary, Pre Schools, High Schools, Junior High Schools, Special, Vocational Schools", "College, University", "Clubs, Community Centers", "Museums", "Offices", "Religious Auditoriums", "Auditoriums", "Sports Arenas/Convention Centers", "Athletic Bldgs", "Libraries", "Cafeterias", "Airport", "Dormitories", "Special, Vocational Schools", "Junior High Schools", "High Schools", "Rental Warehouses", "Laboratories", "Transportation Terminals", "Golf Course / Country Club")))
        OR (pc._Name = "GOVERNMENT" AND EXISTS (SELECT 1 FROM UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) AS sub WHERE sub IN ("Courthouses", "Fire and Police Stations", "Prisons", "Offices", "Government - Misc. Bldgs.", "Government Offices", "Clubs, Community Centers", "Museums", "College, University", "Elementary, Pre Schools, High Schools, Junior High Schools, Special, Vocational Schools", "Airport", "Auditoriums", "Libraries", "Sports Arenas/Convention Centers", "Post Offices", "Rental Warehouses", "Elementary, Pre Schools, Junior High Schools", "Religious Auditoriums", "High Schools", "Athletic Bldgs")))
        OR (pc._Name = "INDUSTRIAL" AND EXISTS (SELECT 1 FROM UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) AS sub WHERE sub IN ("Transportation Terminals", "Golf Course / Country Club", "Offices", "Government - Misc. Bldgs.", "Government Offices", "Elementary, Pre Schools, High Schools, Junior High Schools, Special, Vocational Schools", "Broadcast Studios", "College, University", "Airport", "Elementary, Pre Schools", "Post Offices", "Fire and Police Stations", "Warehouses", "Rental Warehouses", "Clubs, Community Centers", "High Schools", "Laboratories", "Religious Auditoriums", "Special, Vocational Schools", "Museums", "Libraries", "Athletic Bldgs", "Sports Arenas/Convention Centers", "Prisons")))
        OR (pc._Name = "MEDICAL" AND EXISTS (SELECT 1 FROM UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) AS sub WHERE sub IN ("Athletic Bldgs", "College, University", "Hospitals, Clinics", "Offices", "Hospitals, Clinics, Medical Offices", "Broadcast Studios", "Elementary, Pre Schools", "Fire and Police Stations", "Manufacturing", "Auditoriums", "Clubs, Community Centers", "Airport", "Warehouses", "Clubs, Community Centers, Religious Auditoriums", "Rental Warehouses", "Religious Auditoriums", "Government Offices", "Prisons", "Dormitories", "High Schools", "Special, Vocational Schools", "Laboratories", "Medical Offices", "Nursing Homes", "Auditoriums, Libraries", "Government - Misc. Bldgs.", "Sports Arenas/Convention Centers", "Transportation Terminals", "Golf Course / Country Club")))
        OR (pc._Name = "MILITARY" AND EXISTS (SELECT 1 FROM UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) AS sub WHERE sub IN ("Hospitals, Clinics, Medical Offices", "Hospitals, Clinics", "Dormitories", "Offices", "Military - Misc.", "Medical Offices", "Airport", "Military Housing", "Museums", "Military Offices", "Government Offices", "Government - Misc. Bldgs.", "Nursing Homes", "Transportation Terminals")))
        OR (pc._Name = "RESIDENTIAL" AND EXISTS (SELECT 1 FROM UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) AS sub WHERE sub IN ("Transportation Terminals", "Golf Course / Country Club", "Offices", "Government - Misc. Bldgs.", "Government Offices", "Clubs, Community Centers", "Religious Auditoriums", "Auditoriums", "Sports Arenas/Convention Centers", "Rental Warehouses", "Condominiums", "Athletic Bldgs", "College, University", "Manufacturing", "Hospitals, Clinics, Medical Offices", "Airport", "Elementary, Pre Schools", "High Schools", "Apartments", "Broadcast Studios", "Laboratories", "Fire and Police Stations", "Museums", "Junior High Schools", "Special, Vocational Schools", "Medical Offices", "Warehouses", "Courthouses", "Dormitories", "Libraries", "Nursing Homes")))
        OR (pc._Name = "RETAIL" AND EXISTS (SELECT 1 FROM UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) AS sub WHERE sub IN ("Golf Course / Country Club", "Hospitals, Clinics, Medical Offices", "Food Stores", "Hotels", "Offices", "Bank", "Restaurants", "Entertainment", "Auditoriums", "Automotive", "Condominiums", "Government - Misc. Bldgs.", "Government Offices", "Clubs, Community Centers", "Museums", "Airport", "Retail Stores", "College, University", "Apartments", "Broadcast Studios", "Shopping Centers", "Elementary, Pre Schools", "Manufacturing", "Warehouses", "Military - Misc.", "Cafeterias", "Rental Warehouses", "Post Offices", "Dormitories", "Athletic Bldgs", "High Schools", "Fire and Police Stations", "Junior High Schools", "Medical Offices", "Laboratories", "Libraries", "Courthouses", "Religious Auditoriums", "Sports Arenas/Convention Centers", "Nursing Homes")))
    )
)
SELECT * EXCEPT(row_num) FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY ProjectID ORDER BY timeCreated DESC) as row_num
  FROM FilteredData
) WHERE row_num = 1;
'''

df = client.query(query).to_dataframe()
df

,ProjectID,DataSourceID,Title,Stage,URL,UpdateDate,IsProspective,UpdateText,Valuation_Value,Valuation_Currency,...,Details_Detail_ContractConditions,Details,Details_Detail_Subbids,Details_Detail_Quantity_Unitprices,Parameters_Parameter_SingleTradeClassification,timeCreated,sourceFile,Notes_Note,Details_Detail_Status,Numeric_Valuation
0,1001012546,US,Center for Science and Technology - Chapman Un...,General Contractor Award,http://insight.cmdgroup.com/SingleSignOn/Proje...,2020-01-26,False,Updated to General Contractor Award stage,80000000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:04:47.848Z,1.4_Adhoc_DL_FBMSales_XML_20241004_10.xml,[],[],80000000.0
1,1001925257,US,Construction Management at Risk for New Constr...,Construction Documents,http://insight.cmdgroup.com/SingleSignOn/Proje...,2020-09-09,False,"Project reviewed, Stage confirmed as Construct...",200000000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:04:47.848Z,1.4_Adhoc_DL_FBMSales_XML_20241004_10.xml,[],[],200000000.0
2,1001979012,US,Sanford Rocks Center Rapids Medical,Construction Underway,http://insight.cmdgroup.com/SingleSignOn/Proje...,2019-11-27,False,"Project reviewed, Stage confirmed as Construct...",5080000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:04:47.848Z,1.4_Adhoc_DL_FBMSales_XML_20241004_10.xml,[],[],5080000.0
3,1002078090,US,937 Bergen Street / 1036 Dean Street,Construction Underway,http://insight.cmdgroup.com/SingleSignOn/Proje...,2020-01-26,False,Updated to Construction Underway stage,20000000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:04:47.848Z,1.4_Adhoc_DL_FBMSales_XML_20241004_10.xml,[],[],20000000.0
4,1002151290,US,Block 56E,Construction Documents,http://insight.cmdgroup.com/SingleSignOn/Proje...,2021-07-12,False,Updated to Construction Documents stage,20000000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:04:47.848Z,1.4_Adhoc_DL_FBMSales_XML_20241004_10.xml,[],[],20000000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7308,1007119894,US,Construction Management services for Cambrian ...,Biddate Set,http://insight.cmdgroup.com/SingleSignOn/Proje...,2024-03-29,False,Duncan Wisniewski Architecture was added as Ar...,26000000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:04:50.422Z,1.4_Adhoc_DL_FBMSales_XML_20241009_235.xml,[],[],26000000.0
7309,1007128511,US,Cardinal Glennon Children's Hospital - BP 03 F...,General Contractor Award,http://insight.cmdgroup.com/SingleSignOn/Proje...,2024-05-24,False,Updated to General Contractor Award stage,140000000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:04:50.422Z,1.4_Adhoc_DL_FBMSales_XML_20241009_235.xml,[],[],140000000.0
7310,1007134603,US,Murphy USA #3509/#6765/#5507 - Cleveland,Construction Documents,http://insight.cmdgroup.com/SingleSignOn/Proje...,2024-09-18,False,"New Project Location is Beechnut St, Houston, ...",2000000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:04:50.422Z,1.4_Adhoc_DL_FBMSales_XML_20241009_235.xml,[],[],2000000.0
7311,1007136318,US,Klein Intermediate School Renovations and Addi...,Construction Documents,http://insight.cmdgroup.com/SingleSignOn/Proje...,2024-04-19,False,Harrison Kornberg Architects was added as Arch...,10100000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:04:50.422Z,1.4_Adhoc_DL_FBMSales_XML_20241009_235.xml,[],[],10100000.0


In [11]:
def assign_bucket(value):
        try:
            if value >= 100000000:
                return ">100M"
            elif 50000000 <= value < 100000000:
                return "50M-100M"
            elif 25000000 <= value < 50000000:
                return "25M-50M"
            elif 10000000 <= value < 25000000:
                return "10M-25M"
            elif 5000000 <= value < 10000000:
                return "5M-10M"
            elif 2000000 <= value < 5000000:
                return "2M-5M"
            elif 1000000 <= value < 2000000:
                return "1M-2M"
            else:
                return "Under 1M"
        except (ValueError, AttributeError):
            return "Invalid Value"

df['Valuation_Bucket'] = df['Numeric_Valuation'].apply(assign_bucket)

In [12]:
# Dictionary to store the value_counts DataFrames
value_counts_dfs = {}

for col in ['Stage', 'ParentCategories_PrimaryCategoryName', 'State', 'Valuation_Bucket']:
    if col == 'State':
        # For State column, create DataFrame from address state counts
        counts = df['Addresses_Address'].apply(lambda x: x[0]['ns0:StateProvince']).value_counts(normalize=True)
        value_counts_dfs[col] = pd.DataFrame(counts).reset_index()
        value_counts_dfs[col].columns = [col, 'Proportion']
    else:
        # For other columns, create DataFrame from direct value counts
        counts = df[col].value_counts(normalize=True)
        value_counts_dfs[col] = pd.DataFrame(counts).reset_index()
        value_counts_dfs[col].columns = [col, 'Proportion']
value_counts_dfs

{'Stage':                       Stage  Proportion
 0     Construction Underway    0.498154
 1  General Contractor Award    0.198004
 2    Construction Documents    0.149597
 3                  Post Bid    0.115137
 4        Low Bids Announced    0.035690
 5               Biddate Set    0.002735
 6             SUBBIDS: ASAP    0.000684,
 'ParentCategories_PrimaryCategoryName':    ParentCategories_PrimaryCategoryName  Proportion
 0                            Apartments    0.292766
 1                         Retail Stores    0.079174
 2                               Offices    0.064953
 3                   College, University    0.058526
 4                          High Schools    0.054971
 5              Fire and Police Stations    0.043074
 6                       Medical Offices    0.035963
 7                         Nursing Homes    0.035143
 8                    Hospitals, Clinics    0.030630
 9                            Automotive    0.028442
 10                          Restaurant

In [13]:
import pandas as pd

# Create the sample DataFrame
df_sample = df.sample(200, random_state=42)

for col in ['Stage', 'ParentCategories_PrimaryCategoryName', 'State', 'Valuation_Bucket']:
    if col == 'State':
        # For State column, create DataFrame from address state counts
        counts = df_sample['Addresses_Address'].apply(lambda x: x[0]['ns0:StateProvince']).value_counts(normalize=True)
        value_counts_dfs[f'{col}_sample'] = pd.DataFrame(counts).reset_index()
        value_counts_dfs[f'{col}_sample'].columns = [col, 'Proportion']
    else:
        # For other columns, create DataFrame from direct value counts
        counts = df_sample[col].value_counts(normalize=True)
        value_counts_dfs[f'{col}_sample'] = pd.DataFrame(counts).reset_index()
        value_counts_dfs[f'{col}_sample'].columns = [col, 'Proportion']

In [14]:
for col in ['Stage', 'ParentCategories_PrimaryCategoryName', 'State', 'Valuation_Bucket']:
    # Merge the DataFrames for the full and
    comparison = value_counts_dfs[col].merge(value_counts_dfs[f'{col}_sample'], on=col, suffixes=('', '_sample'))
    comparison['Proportion_diff'] = comparison['Proportion'] - comparison['Proportion_sample']
    print(f'Comparison of {col} value counts:')
    print(comparison)
    print('\n')

    print(f'Summary statistics for the difference in proportions:')
    print(comparison['Proportion_diff'].apply(lambda x: abs(x)).describe())
    print('\n---------------------------------------------------\n')

Comparison of Stage value counts:
                      Stage  Proportion  Proportion_sample  Proportion_diff
0     Construction Underway    0.498154              0.535        -0.036846
1  General Contractor Award    0.198004              0.190         0.008004
2    Construction Documents    0.149597              0.125         0.024597
3                  Post Bid    0.115137              0.125        -0.009863
4        Low Bids Announced    0.035690              0.025         0.010690


Summary statistics for the difference in proportions:
count    5.000000
mean     0.018000
std      0.012433
min      0.008004
25%      0.009863
50%      0.010690
75%      0.024597
max      0.036846
Name: Proportion_diff, dtype: float64

---------------------------------------------------

Comparison of ParentCategories_PrimaryCategoryName value counts:
   ParentCategories_PrimaryCategoryName  Proportion  Proportion_sample  \
0                            Apartments    0.292766              0.265   
1    

In [15]:
df_sample.to_csv("data/curated_data/sample_data.csv")